# Step 8: evaluate a fine-tuned adapter on dev (prompt v2) -- Exp 4 (Run A)

Loads base Qwen2.5-Coder-1.5B-Instruct in fp16, attaches a LoRA adapter downloaded from a
private Hugging Face repo, merges it into the base weights, and runs it on all 1,034 dev
prompts (prompt v2 -- the Step 4+5 winner), using the **same decoding settings as the
baseline** (greedy, `max_new_tokens=256`, stop at `<|im_end|>`) so the comparison is fair.

Inference runs in fp16, not 4-bit: the base model is only 1.5B parameters (fits easily in a
T4's 16GB at fp16), and evaluating the merged adapter at full precision avoids any
quantization-specific generation differences between this and the fp16 baseline run --
QLoRA's 4-bit loading was a *training*-time memory trick (Step 7), not a requirement for
inference.

**Before Run All:**
- Settings -> Accelerator: **GPU T4 x2** (only one T4 used, intentional), Internet: **On**.
- Add Input -> the same prompts dataset uploaded for Step 4+5 (needs `prompts_v2_dev.jsonl`).
- Add-ons -> Secrets: **`HF_TOKEN`** (read access is enough -- it only downloads the
  adapter from a private repo here, doesn't push anything).
- Edit `ADAPTER_REPO_ID` / `ADAPTER_PATH_IN_REPO` / `TAG` in the config cell if evaluating
  a different run (e.g. Run B) -- defaults are set for Run A.

**What this notebook does:**
- Same environment guard and `safe_pip_install` dry-run-before-install pattern as Step 7
  (never risk the on-disk torch/CUDA build -- see the vLLM lesson in PROGRESS.md).
- Downloads the adapter from `ADAPTER_REPO_ID/ADAPTER_PATH_IN_REPO` on the HF Hub, checks
  its `adapter_config.json` actually points back at the expected base model before loading
  it (fails loudly on a mismatch rather than silently loading the wrong thing), attaches it
  with PEFT, and merges it into the base weights.
- Batched HF `generate`, fp16, left-padded, greedy decoding, `max_new_tokens=256`, stops at
  the model's real end-of-turn token (`<|im_end|>`), length-sorted batching restored to
  original order, automatic batch-size halving on a CUDA OOM -- identical generation code
  to the Step 4+5 baseline notebook, so any accuracy difference comes from the adapter, not
  from a different decoding setup.
- **Resumable** the same way as Step 4+5: output flushed every 32 examples; a prior partial
  or complete `predictions_<TAG>.jsonl` attached via Notebook Output Files is detected and
  continued/skipped.
- Output: `/kaggle/working/predictions_<TAG>.jsonl` (same schema as the baseline's
  predictions files -- idx, db_id, question, raw_output, gen_time_sec -- so
  `scripts/evaluate_experiment.py` works on it unmodified) and `run_metadata.json`.

**IMPORTANT -- use Save Version -> Save & Run All, not an interactive Run All.** An
interactive-only run's `/kaggle/working` is wiped when the session ends, which is exactly
what happened to Run A's training output (see PROGRESS.md). Only a saved Version's output
files survive and can be downloaded or attached later.

## How to resume an interrupted run
1. Make sure the run you want to resume from was saved with **Save Version -> Save & Run All**.
2. Open this notebook again (new session is fine).
3. **Add Input** -> **Notebook Output Files** tab -> find this notebook -> pick the version
   to resume from -> **Add**. Keep the prompts dataset attached too.
4. **Run All** again. The notebook detects `predictions_<TAG>.jsonl` under `/kaggle/input/...`,
   copies it into `/kaggle/working`, prints `RESUMING`, and continues from the first row
   that hasn't been generated yet.


In [ ]:
# Environment check -- catch a broken environment before spending any GPU time.
import json as _json
import os
import subprocess
import sys

print("Python:", sys.version)

import torch
print("torch:", torch.__version__, "| torch's CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU available in this session. Check Settings -> Accelerator is set to GPU T4 x2, "
        "then Run All again."
    )

print("GPU:", torch.cuda.get_device_name(0))

import transformers
print("transformers:", transformers.__version__)

# Packages a pip install must never be allowed to touch -- this is exactly what broke an
# earlier notebook's vLLM install (pip silently upgraded torchaudio to a build that didn't
# match the already-loaded torch).
WATCHED_PACKAGES = ["torch", "torchvision", "torchaudio", "triton", "transformers"]


def installed_torch_build():
    """What's actually on disk right now, read via a FRESH subprocess -- not the
    already-imported `torch` module in this kernel, which can't reflect a pip install
    that happened after it was first imported."""
    r = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.__version__); print(torch.version.cuda)"],
        capture_output=True, text=True,
    )
    lines = r.stdout.strip().splitlines()
    return (lines[0], lines[1]) if len(lines) >= 2 else (None, None)


TORCH_BUILD_AT_START = installed_torch_build()
print("torch build on disk, recorded for later comparison:", TORCH_BUILD_AT_START)


def _current_versions(names):
    out = {}
    for name in names:
        r = subprocess.run(
            [sys.executable, "-c", f"import importlib.metadata as m; print(m.version('{name}'))"],
            capture_output=True, text=True,
        )
        out[name] = r.stdout.strip() if r.returncode == 0 else None
    return out


def dry_run_would_change_watched(args):
    """Simulate the install with `pip install --dry-run --report <file>` -- installs
    nothing -- and check whether the packages it WOULD install/upgrade include any of
    WATCHED_PACKAGES at a version different from what's currently on disk. If anything
    would change, the real install never runs at all."""
    report_path = os.path.join(OUTPUT_DIR, "_pip_dry_run_report.json")
    if os.path.exists(report_path):
        os.remove(report_path)
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--dry-run", "--quiet",
         "--report", report_path] + args,
        capture_output=True, text=True,
    )
    if result.returncode != 0 or not os.path.exists(report_path):
        return True, {"error": (result.stderr or "no dry-run report produced").strip()[:500]}

    with open(report_path, encoding="utf-8") as f:
        report = _json.load(f)

    current = _current_versions(WATCHED_PACKAGES)
    changes = {}
    for item in report.get("install", []):
        meta = item.get("metadata", {})
        pkg_name = meta.get("name", "")
        pkg_version = meta.get("version")
        for watched in WATCHED_PACKAGES:
            if pkg_name.lower() == watched.lower() and pkg_version != current.get(watched):
                changes[watched] = {"current": current.get(watched), "would_become": pkg_version}
    return (len(changes) > 0), changes


def safe_pip_install(args, label):
    """Install `args`, but only after a dry run confirms it wouldn't touch any
    WATCHED_PACKAGES. Returns True if installed (or already satisfied), False if skipped."""
    would_change, details = dry_run_would_change_watched(args)
    if would_change:
        print(f"{label}: SKIPPING install -- a dry run shows it would change a protected "
              f"package: {details}. Nothing was installed.")
        return False

    before = installed_torch_build()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + args, check=True)
    after = installed_torch_build()
    if after != before:
        raise RuntimeError(
            f"{label}: installing it changed the on-disk torch build from {before} to "
            f"{after} even though the dry-run check passed. Stopping the notebook rather "
            f"than continue on a possibly broken environment."
        )
    return True


In [ ]:
# ---- Config -- edit ADAPTER_REPO_ID / TAG to evaluate a different run ----
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # single T4 on purpose

BASE_MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_REPO_ID = "CHANGE_ME/text2sql-qlora-run-a"  # <-- CHANGE ME if not FARHANFAST's repo
ADAPTER_PATH_IN_REPO = "final_adapter"
TAG = "run_a"  # output file becomes predictions_run_a.jsonl -- change for Run B, etc.

SEED = 0
MAX_NEW_TOKENS = 256
MAX_MODEL_LEN = 4096      # matches the Step 7 training cutoff
CHUNK_SIZE = 32            # examples per flush/resume checkpoint
INITIAL_BATCH_SIZE = 8     # halves automatically and retries on CUDA OOM

OUTPUT_DIR = OUT_DIR = "/kaggle/working"


In [ ]:
# peft isn't preinstalled on Kaggle -- only install it if missing, and only after
# confirming (dry run) it wouldn't touch a watched package.
try:
    import peft
    print("peft already installed:", peft.__version__)
except ImportError:
    ok = safe_pip_install(["peft"], "peft")
    if not ok:
        raise RuntimeError("peft is required to load the adapter and its install was "
                            "blocked by the dry-run safety check -- see the message above.")
    import peft
    print("peft installed:", peft.__version__)


In [ ]:
# Kaggle secrets
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except Exception:
    raise RuntimeError("Missing Kaggle secret HF_TOKEN. Add-ons -> Secrets -> add HF_TOKEN "
                        "(read access is enough), then Run All again.")
print("secret loaded OK")


In [ ]:
# Find the dev prompts under /kaggle/input (recursive search, never a fixed dataset slug).
import glob


def find_input_file(filename):
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    return matches[0] if matches else None


PROMPTS_FNAME = "prompts_v2_dev.jsonl"
PROMPTS_PATH = find_input_file(PROMPTS_FNAME)
if PROMPTS_PATH is None:
    raise FileNotFoundError(
        f"Missing {PROMPTS_FNAME} under /kaggle/input. Attach the same prompts dataset "
        "used for Step 4+5 (or re-upload data/processed/kaggle_upload/prompts_v2_dev.jsonl)."
    )
print(f"found {PROMPTS_FNAME} -> {PROMPTS_PATH}")


In [ ]:
import shutil

OUT_PATH = os.path.join(OUT_DIR, f"predictions_{TAG}.jsonl")

print("Checking for a previous run to resume from...")
if os.path.exists(OUT_PATH):
    print(">>> predictions file already present this session -- leaving it alone.")
else:
    prev_path = find_input_file(f"predictions_{TAG}.jsonl")
    if prev_path is not None:
        shutil.copy(prev_path, OUT_PATH)
        print(f">>> RESUMING: copied previous output {prev_path} -> {OUT_PATH}")
    else:
        print(">>> STARTING FRESH -- no previous run output found under /kaggle/input.")


In [ ]:
# Download the adapter from the private HF repo, and check it actually points back at
# the base model we're about to load it onto before trusting it -- a mismatched adapter
# would load "successfully" and silently produce garbage or near-random generations.
import json
from huggingface_hub import HfApi, snapshot_download

api = HfApi(token=HF_TOKEN)
repo_info = api.repo_info(repo_id=ADAPTER_REPO_ID)
print(f"adapter repo {ADAPTER_REPO_ID} visibility: {'private' if repo_info.private else 'PUBLIC'}")

ADAPTER_LOCAL_DIR = os.path.join(OUTPUT_DIR, "adapter")
snapshot_download(
    repo_id=ADAPTER_REPO_ID, local_dir=ADAPTER_LOCAL_DIR,
    allow_patterns=f"{ADAPTER_PATH_IN_REPO}/*", token=HF_TOKEN,
)
ADAPTER_DIR = os.path.join(ADAPTER_LOCAL_DIR, ADAPTER_PATH_IN_REPO)

with open(os.path.join(ADAPTER_DIR, "adapter_config.json"), encoding="utf-8") as f:
    adapter_config = json.load(f)

adapter_base = adapter_config.get("base_model_name_or_path")
if adapter_base != BASE_MODEL_NAME:
    raise ValueError(
        f"Adapter at {ADAPTER_REPO_ID}/{ADAPTER_PATH_IN_REPO} was trained against base "
        f"model '{adapter_base}', not '{BASE_MODEL_NAME}' -- refusing to attach it to the "
        f"wrong base. Fix BASE_MODEL_NAME (or ADAPTER_REPO_ID) in the config cell."
    )
print(f"adapter base model matches: {adapter_base}")
print(f"adapter LoRA config: r={adapter_config.get('r')}, alpha={adapter_config.get('lora_alpha')}, "
      f"target_modules={adapter_config.get('target_modules')}")


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # required for correct batched causal-LM generation

# Qwen's chat template ends a turn with <|im_end|>, which may not be the tokenizer's
# default eos_token_id -- stop generation at both, same as the Step 4+5 baseline.
eos_ids = {tokenizer.eos_token_id}
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
if im_end_id is not None and im_end_id != getattr(tokenizer, "unk_token_id", None):
    eos_ids.add(im_end_id)
eos_ids = sorted(eos_ids)
print("stopping generation at token ids:", eos_ids, [tokenizer.decode([i]) for i in eos_ids])

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME, torch_dtype=torch.float16).to("cuda")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model = model.merge_and_unload()  # fold the adapter into the base weights for plain generate()
model.eval()
print("base model + adapter loaded and merged on", next(model.parameters()).device)


In [ ]:
def generate_batch_safe(prompts, batch_size):
    """Generate for `prompts`, halving batch_size and retrying on a CUDA OOM instead of crashing."""
    try:
        enc = tokenizer(prompts, return_tensors="pt", padding=True, truncation=True,
                        max_length=MAX_MODEL_LEN).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                  eos_token_id=eos_ids, pad_token_id=tokenizer.pad_token_id)
        gen_only = out[:, enc["input_ids"].shape[1]:]
        return tokenizer.batch_decode(gen_only, skip_special_tokens=True)
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        torch.cuda.empty_cache()
        if batch_size <= 1:
            raise
        half = batch_size // 2
        print(f"  CUDA OOM at batch_size={batch_size}, halving to {half} and retrying")
        results = []
        for i in range(0, len(prompts), half):
            results.extend(generate_batch_safe(prompts[i:i + half], half))
        return results


def generate_chunk(rows, batch_size):
    """rows: list of {idx, db_id, question, prompt}. Returns raw output strings in the
    SAME order as `rows` -- sorts by prompt length internally, restores original order."""
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]["prompt"]))
    sorted_prompts = [rows[i]["prompt"] for i in order]
    sorted_outputs = []
    for i in range(0, len(sorted_prompts), batch_size):
        batch = sorted_prompts[i:i + batch_size]
        sorted_outputs.extend(generate_batch_safe(batch, batch_size))
    outputs = [None] * len(rows)
    for pos, orig_idx in enumerate(order):
        outputs[orig_idx] = sorted_outputs[pos]
    return outputs


In [ ]:
import time

def load_prompts(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def load_done(path):
    done = {}
    if not os.path.exists(path):
        return done
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            done[row["idx"]] = row
    return done


rows = load_prompts(PROMPTS_PATH)
done = load_done(OUT_PATH)

run_metadata = {
    "experiment": f"Exp 4: evaluate adapter ({TAG})",
    "base_model": BASE_MODEL_NAME, "adapter_repo": ADAPTER_REPO_ID,
    "adapter_path_in_repo": ADAPTER_PATH_IN_REPO, "adapter_config": adapter_config,
    "prompt_version": "v2", "seed": SEED,
    "max_new_tokens": MAX_NEW_TOKENS, "decoding": "greedy", "eos_token_ids": eos_ids,
    "python_version": sys.version, "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda, "transformers_version": transformers.__version__,
    "peft_version": peft.__version__, "gpu_name": torch.cuda.get_device_name(0),
}

if len(done) >= len(rows):
    print(f"{TAG}: already complete ({len(done)}/{len(rows)}) -- skipping generation")
    run_metadata["status"] = "already complete, skipped"
else:
    with open(OUT_PATH, "w", encoding="utf-8") as f:
        for idx in sorted(done):
            f.write(json.dumps(done[idx]) + "\n")

    remaining = [r for r in rows if r["idx"] not in done]
    if done:
        print(f"{TAG}: RESUMING -- {len(done)}/{len(rows)} already done, {len(remaining)} remaining")
    else:
        print(f"{TAG}: STARTING FRESH -- {len(remaining)} to do")

    run_start = time.time()
    with open(OUT_PATH, "a", encoding="utf-8") as f:
        for i in range(0, len(remaining), CHUNK_SIZE):
            chunk = remaining[i:i + CHUNK_SIZE]
            t0 = time.time()
            raw_outputs = generate_chunk(chunk, INITIAL_BATCH_SIZE)
            elapsed = time.time() - t0
            per_example_time = elapsed / len(chunk)
            for r, raw in zip(chunk, raw_outputs):
                f.write(json.dumps({
                    "idx": r["idx"], "db_id": r["db_id"], "question": r["question"],
                    "raw_output": raw, "gen_time_sec": per_example_time,
                }) + "\n")
            f.flush()
            os.fsync(f.fileno())
            done_so_far = len(done) + i + len(chunk)
            print(f"{TAG}: {done_so_far}/{len(rows)} done, last chunk {elapsed:.1f}s")
    total_elapsed = time.time() - run_start
    run_metadata["n_examples"] = len(rows)
    run_metadata["newly_generated"] = len(remaining)
    run_metadata["total_time_sec"] = total_elapsed
    print(f"=== {TAG} done in {total_elapsed:.1f}s, wrote {OUT_PATH} ===")

with open(os.path.join(OUT_DIR, "run_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(run_metadata, f, indent=2)
print(json.dumps({k: v for k, v in run_metadata.items() if k != "adapter_config"}, indent=2))


## Download these files from /kaggle/working/ when done:
- `predictions_run_a.jsonl` (or `predictions_<TAG>.jsonl` if evaluating a different run)
- `run_metadata.json`

Put `predictions_run_a.jsonl` at `results/step8_run_a/predictions_run_a.jsonl` locally,
then run:

```
python scripts/evaluate_experiment.py --pred results/step8_run_a/predictions_run_a.jsonl --tag exp4_run_a
```

If you need to stop partway through, use **Save Version -> Save & Run All** before
stopping, so the partial output is attachable later (see "How to resume" above).